<h4 style="color:white;text-align:center; background-color:#4CAF50; padding:5px; border-radius:5px;">
Purpose of ItemBased Recommendation
</h4>

In [7]:
from IPython.display import display, HTML
display(HTML(f"""
<div>
    <h3>Recommendation Result</h3>
    <p>Input Movie: 10</p>
    <p>Output users: [3, 4]</p>
    <p>Recommend Movie 10 → to users [3, 4]</p>
</div>
"""))

### Load the Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances

#### Load the Dataset

In [3]:
# reading ratings file:

# Define column names for the dataset
# These names will be assigned to each column in the file
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

# Read the dataset file into a pandas DataFrame
'ml-100k/u.data', # Path to the dataset file
'sep=\t',  # Data is separated by TAB (not comma)
'names=r_cols,' # Assign column names defined above
'encoding=latin-1' # Handle special characters(latin-1) properly while reading file  

ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols,encoding='latin-1')

In [4]:
# Now we give movieid respective movie list
# Define column names for the movie dataset
# Includes movie_id, title, release info, and genre columns etc
i_cols = [
    'movie id', 'movie title', 'release date', 'video release date',
    'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation',
    'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama',
    'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

# Read movie dataset from file
# sep='|' → values separated by pipe symbol
# names=i_cols → assign column names
# encoding='latin-1' → handle special characters properly
items = pd.read_csv('ml-100k/u.item',sep='|',names=i_cols, encoding='latin-1')

#### Final Movie List for the users

In [5]:
def itemBasedRecommendation(ratings, items, item_input,similar_item_count=5,top_n=10,thres=0.1):

    # Step 1: Create User-Item Matrix
    # Rows = Users, Columns = Movies, Values = Ratings
    data_matrix = ratings.pivot_table(
        index='user_id',
        columns='movie_id',
        values='rating'
    ).fillna(0)


    # Step 2: Compute Item Similarity
    # Convert distance → similarity using cosine
    item_similarity = 1 - pairwise_distances(
        data_matrix.T,
        metric='cosine'
    )

    # Convert to DataFrame
    item_sim_df = pd.DataFrame(
        item_similarity,
        index=data_matrix.columns,
        columns=data_matrix.columns
    )


    # Step 3: Predict Ratings
    similarity_sum = np.abs(item_similarity).sum(axis=1)

    pred = data_matrix.values.dot(item_similarity)
    pred = pred / similarity_sum

    pred_df = pd.DataFrame(
        pred,
        index=data_matrix.index,
        columns=data_matrix.columns
    )


    # Step 4: Check if input movie exists
    if item_input not in item_sim_df.columns:
        return []


    # Step 5: Get similar items to input movie
    similar_items = item_sim_df[item_input].sort_values(
        ascending=False
    )[1:similar_item_count + 1]


    # Step 6: Get users who watched similar movies
    similar_users = ratings[
        ratings['movie_id'].isin(similar_items.index)
    ]['user_id'].unique()


    # Step 7: Get users who already watched input movie
    input_users = ratings[
        ratings['movie_id'] == item_input
    ]['user_id'].unique()


    # Step 8: Filter users
    # Keep users who watched similar items BUT not input movie
    final_users = list(set(similar_users) - set(input_users))

    if len(final_users) == 0:
        return []


    # Step 9: Predict score for input movie
    user_scores = pred_df.loc[final_users, item_input]


    # Step 10: Apply threshold
    user_scores = user_scores[user_scores >= thres]


    # Step 11: Sort and select top users
    top_users = user_scores.sort_values(ascending=False).head(top_n)


    # Step 12: Convert to list
    recommended_users = top_users.index.tolist()


    # Final output
    return recommended_users

In [6]:
recommended = itemBasedRecommendation(ratings=ratings,items=items,item_input=50,
    similar_item_count=5,top_n=10,thres=0.1)

print("Recommended to Users:")
for u in recommended:
    print(u)

Recommended to Users:
90
532
932
788
314
207
16
543
342
627
